<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2009%20-%20Describing%20Data%3A%20Mean%2C%20Variance%2C%20Distributions/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 09 — Describing Data: Mean, Variance, Distributions · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

Ten *identical* 3-room, 900 sq ft flats on one street sold for (₹ lakh):

$$9,\ 10,\ 11,\ 11,\ 12,\ 12,\ 12,\ 13,\ 14,\ 16$$

Same inputs, different outputs. No model of the form $\hat y = \mathbf{w}\cdot\mathbf{x}+b$
can produce ten different numbers from one input — and this time there is no missing feature
to blame. Real measurements scatter.

## Step 2 — Prediction

Commit before running. You check these in Step 10.

1. Add up the ten deviations from the mean. What do you get — and would a different dataset
   give something different?
2. Which is larger for this data: the mean absolute deviation, or the standard deviation?
3. Replace the ₹16 lakh sale with ₹60 lakh. What happens to the mean, the median, and the
   standard deviation?
4. Chapter 5 computed the correlation between rooms and area as $0.70232$. Chapter 2 defined
   cosine similarity. Are those two formulas related?

In [ ]:
# Step 3 — Intuition: the mean is what each house would have cost if the total were shared.
import numpy as np
np.random.seed(0)

prices = np.array([9., 10., 11., 11., 12., 12., 12., 13., 14., 16.])
n = len(prices)

print("total  =", prices.sum())
print("shared =", prices.sum() / n, "per house")
print("numpy  =", prices.mean())
assert prices.sum() == 120.0
assert prices.mean() == 12.0

# Range: identical houses, seven lakh apart.
print(f"\ncheapest {prices.min()}, dearest {prices.max()}, spread {prices.max()-prices.min()}")

## Step 4 — The Mathematics Under Test

$$\bar x = \frac1n\sum_i x_i
\qquad
\sum_i (x_i - \bar x) = 0
\qquad
s^2 = \frac{1}{n-1}\sum_i (x_i - \bar x)^2$$

$$r = \frac{\mathrm{cov}(x,y)}{s_x s_y}
\qquad
p(x) \propto \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

In [ ]:
# Step 5 — Manual calculation: deviations, and the trap they set (blog sections 4-5).
dev = prices - prices.mean()
print("deviations:", dev)
print("their sum :", dev.sum())
assert dev.sum() == 0.0          # EXACTLY zero, not approximately

# So "average the deviations" reports zero spread for every dataset ever.
print("\naverage deviation =", dev.mean(), "  <- useless, always")
assert dev.mean() == 0.0

# This is Chapter 1 section 7 again: a model quoting 6 lakh for every house
# scored a perfect average error of zero because +3,+1 cancelled -1,-3.
old = np.array([3., 1., -1., -3.])
assert old.mean() == 0.0
print("Chapter 1's cancellation, same disease:", old, "-> mean", old.mean())

In [ ]:
# Step 5b — remove the sign two ways (blog section 6).
mad = np.abs(dev).mean()
sq = dev ** 2
print("squared deviations:", sq)
print("their sum         :", sq.sum())
assert sq.sum() == 36.0

var_pop = sq.sum() / n            # describes THESE ten houses
var_sample = sq.sum() / (n - 1)   # estimates the whole street
sd = np.sqrt(var_sample)

print(f"\nMAD              = {mad}")
print(f"variance (/n)    = {var_pop}")
print(f"variance (/n-1)  = {var_sample}")
print(f"std deviation    = {sd}   <- back in rupees lakh")
assert mad == 1.4
assert var_pop == 3.6 and var_sample == 4.0 and sd == 2.0
assert np.isclose(sd, prices.std(ddof=1))     # numpy agrees when told ddof=1
assert mad < sd                                # MAD is always the smaller one

print(f"\nSo the quote is: about {prices.mean()} lakh, give or take {sd} lakh.")

In [ ]:
# Step 5c — the computational shortcut from the Level 3 exercise.
# (1/n) sum (x - xbar)^2  ==  (1/n) sum x^2  -  xbar^2
lhs = ((prices - prices.mean()) ** 2).mean()
rhs = (prices ** 2).mean() - prices.mean() ** 2
print(f"sum of x^2 = {(prices**2).sum()}")
print(f"left  form = {lhs}")
print(f"right form = {rhs}")
assert (prices ** 2).sum() == 1476.0
assert np.isclose(lhs, rhs)

# The right form needs one pass, but subtracts two nearly equal numbers --
# exactly Chapter 6 section 11's cancellation problem. Try it with huge values:
big = prices + 1e8
lhs_big = ((big - big.mean()) ** 2).mean()
rhs_big = (big ** 2).mean() - big.mean() ** 2
print(f"\nshifted by 1e8:  left = {lhs_big:.6f}   right = {rhs_big:.6f}")
print("The one-pass form loses precision badly. The mathematics is identical;")
print("the arithmetic is not.")

In [ ]:
# Step 6 — First implementation: correlation IS Chapter 2's cosine similarity.
rooms = np.array([2., 2., 3., 4.])
area  = np.array([800., 1200., 900., 1600.])

def covariance(x, y):
    return ((x - x.mean()) * (y - y.mean())).sum() / (len(x) - 1)

def correlation(x, y):
    return covariance(x, y) / (x.std(ddof=1) * y.std(ddof=1))

def cosine(u, v):                          # Chapter 2, section 9
    return (u @ v) / (np.sqrt(u @ u) * np.sqrt(v @ v))

r = correlation(rooms, area)
print(f"covariance   = {covariance(rooms, area):.4f}   (units: room x sq ft -- meaningless)")
print(f"correlation  = {r:.5f}   (unitless)")
assert np.isclose(covariance(rooms, area), 241.6667, atol=1e-3)
assert np.isclose(r, 0.70232, atol=1e-5)
assert np.isclose(r, np.corrcoef(rooms, area)[0, 1])

# The same number, from Chapter 2's formula applied to the CENTRED vectors:
cos = cosine(rooms - rooms.mean(), area - area.mean())
print(f"cosine of centred vectors = {cos:.5f}")
assert np.isclose(r, cos)
print(f"\nangle between them = {np.degrees(np.arccos(cos)):.1f} degrees")
print("Chapter 5's PCA used this exact 0.70232 in its correlation matrix.")

In [ ]:
# Step 7 — Visualization: the shape of the scatter.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) histogram with mean and +-1 sd
axes[0].hist(prices, bins=np.arange(8.5, 17.5, 1), edgecolor="white")
axes[0].axvline(prices.mean(), color="crimson", lw=2, label=f"mean {prices.mean():.1f}")
axes[0].axvspan(prices.mean()-sd, prices.mean()+sd, alpha=.15, color="crimson",
                label=f"$\\pm$1 sd ({sd:.1f})")
axes[0].set_title("ten identical flats"); axes[0].set_xlabel("price (lakh)")
axes[0].legend(fontsize=8)

# (b) the Gaussian whose exponent is squared distance / variance
xs = np.linspace(5, 19, 300)
gauss = np.exp(-((xs - 12) ** 2) / (2 * 4.0)) / np.sqrt(2 * np.pi * 4.0)
axes[1].plot(xs, gauss, lw=2)
axes[1].axvline(12, color="crimson", ls="--", lw=1)
axes[1].set_title("Gaussian: exponent is $-(x-\\mu)^2/2\\sigma^2$")
axes[1].set_xlabel("price (lakh)")

# (c) the outlier's effect
spoiled = prices.copy(); spoiled[-1] = 60.
labels = ["mean", "median", "std dev"]
before = [prices.mean(), np.median(prices), prices.std(ddof=1)]
after = [spoiled.mean(), np.median(spoiled), spoiled.std(ddof=1)]
xpos = np.arange(3)
axes[2].bar(xpos - .2, before, .4, label="original")
axes[2].bar(xpos + .2, after, .4, label="one mansion added")
axes[2].set_xticks(xpos); axes[2].set_xticklabels(labels)
axes[2].set_title("only the median survives"); axes[2].legend(fontsize=8)

for ax in axes: ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Step 8 — The experiment: one number changes everything (blog section 9).
spoiled = prices.copy()
spoiled[-1] = 60.0                      # one mansion in the street's records

print(f"{'':>16} {'original':>12} {'with mansion':>14}")
for name, f in [("mean", np.mean),
                ("median", np.median),
                ("std dev", lambda a: np.std(a, ddof=1))]:
    print(f"{name:>16} {f(prices):>12.2f} {f(spoiled):>14.2f}")

assert np.isclose(spoiled.mean(), 16.4)
assert np.median(spoiled) == 12.0        # completely unmoved
assert np.isclose(spoiled.std(ddof=1), 15.385, atol=1e-3)

# Why: the squared deviation of the outlier dwarfs everything else.
dev_bad = spoiled - spoiled.mean()
share = dev_bad[-1] ** 2 / (dev_bad ** 2).sum()
print(f"\nthe single mansion accounts for {100*share:.1f}% of the total squared deviation")
assert share > 0.85
print("The mean is now higher than 9 of the 10 houses actually sold for.")

In [ ]:
# Step 9 — Change exactly one variable: the sample size.
# Each sample of n houses gives its own mean. How much do those means scatter?
rng = np.random.default_rng(0)
TRUE_MU, TRUE_SIGMA = 12.0, 2.0

print(f"{'n':>6} {'spread of the sample mean':>28} {'sigma/sqrt(n)':>16}")
for sample_n in [10, 40, 160, 640]:
    means = [rng.normal(TRUE_MU, TRUE_SIGMA, sample_n).mean() for _ in range(4000)]
    observed = np.std(means, ddof=1)
    predicted = TRUE_SIGMA / np.sqrt(sample_n)
    print(f"{sample_n:>6} {observed:>28.4f} {predicted:>16.4f}")
    assert abs(observed - predicted) < 0.02

# Quadrupling the sample halves the uncertainty: the sqrt(n) law.
needed = (TRUE_SIGMA / 0.1) ** 2
print(f"\nto pin the average price to +-0.1 lakh you need about {needed:.0f} sales")
assert np.isclose(needed, 400)

## Step 10 — Observe

Against your Step 2 predictions:

1. The deviations sum to **exactly** zero, and always will — §4 proved it follows from the
   definition of the mean. So "average the deviations" is not a weak measure of spread, it is
   an empty one.
2. MAD $= 1.4$ is **smaller** than $s = 2.0$, and always is: squaring inflates the large
   deviations before averaging.
3. The mansion moves the mean to $16.4$ — higher than nine of the ten actual sales — and the
   standard deviation to $15.4$. The median does not move at all. That single house accounts
   for over 85% of the total squared deviation.
4. Yes: correlation is *exactly* Chapter 2's cosine similarity applied to the centred
   vectors. Both give $0.70232$, an angle of about $45°$ — the same number Chapter 5's PCA used.

Step 9 adds one more: the sample mean's own spread is $\sigma/\sqrt{n}$, so quadrupling the
data only halves the uncertainty.

## Step 11 — Explain

**Why deviations must cancel.** $\sum(x_i - \bar x) = \sum x_i - n\bar x = n\bar x - n\bar x = 0$.
It is the definition of the mean rearranged, which is why no dataset escapes it. Chapter 1 hit
this exact wall with signed errors; the cure is the same — remove the sign.

**Why $n-1$.** Once the deviations must sum to zero, knowing nine of them determines the tenth.
Ten numbers, nine independent pieces of information about spread — one was spent computing
$\bar x$. Dividing by $n-1$ corrects for that.

**Why the outlier wrecks the standard deviation but not the median.** Variance sums *squared*
deviations, so a value 44 away contributes $44^2 = 1936$ — fifty times the other nine combined.
The median only asks which value sits in the middle, so the magnitude of the largest number is
irrelevant to it.

> Squaring's great strength is its great weakness. Chapter 1 chose squared error *because* it
> punishes large mistakes disproportionately. That is the same property that lets one mansion
> dominate a variance — and, in Chapter 13, lets one mislabelled house drag a fitted line.

**Why the Gaussian matters here.** Its exponent is $-(x-\mu)^2/2\sigma^2$ — a squared deviation
divided by the variance. That is the quantity we invented in §6 and the quantity Chapter 1
chose as a loss, arrived at from three unrelated directions. Chapter 12 shows why.

In [ ]:
# Step 12 — Challenges.

# LEVEL 2 (by hand first): for [10, 12, 12, 14, 17], compute mean, all five deviations,
# both variances, both roots, and the MAD. Then check:
small = np.array([10., 12., 12., 14., 17.])
print("mean:", small.mean(), " sum of deviations:", (small - small.mean()).sum())
print("var(/n):", small.var(), " var(/n-1):", small.var(ddof=1))
print("MAD:", np.abs(small - small.mean()).mean(), " sd:", small.std(ddof=1))
assert np.isclose((small - small.mean()).sum(), 0.0)

# LEVEL 4 (Investigate): the sqrt(n) law appeared in Step 9. Now test whether it
# still holds when the data is NOT Gaussian -- try a heavy-tailed distribution
# (e.g. rng.standard_t(df=2, size=n) scaled) and see whether the sample mean
# still tightens like 1/sqrt(n). Explain what you find using section 7's caveats
# about the Central Limit Theorem.

# YOUR CODE HERE


# LEVEL 5 (Design): build a spread measure that one mansion cannot wreck.
# Verify it on `spoiled`, then state honestly what it loses versus the standard deviation.
def robust_spread(x):
    # YOUR CODE HERE
    ...

## Step 13 — Reflection

- [ ] I can explain why averaging deviations always returns zero, without looking it up.
- [ ] I can say why we divide by $n-1$ in terms of degrees of freedom.
- [ ] I can state what the standard deviation means in the data's own units.
- [ ] I showed that correlation and cosine similarity are the same formula.
- [ ] I can explain why the median survived the mansion and the mean did not.
- [ ] I can point at the squared deviation inside the Gaussian's exponent.
- [ ] I know what a "noise floor" is and why beating it means memorizing.

### The question this chapter leaves open

Every tool here is **backward-looking** — it describes numbers we already have. Now the
manager asks: *"a buyer offered ₹14 lakh — what are the chances the next flat beats that?"*

The mean is 12 and the standard deviation 2, so 14 is one standard deviation above the middle.
But "one standard deviation above" is not a chance. Two of our ten sales beat 14, so perhaps
20% — except that is a fact about ten sales that already happened, and the question is about
one that has not.

➡️ **Next:** [Chapter 10 — Probability: Reasoning Under Uncertainty](<../Lecture 10 - Probability: Reasoning Under Uncertainty/blog.md>)